# 01 — Data Exploration

Explore the datasets used for VerifyAI fake news detection:
1. **LIAR** — 12.8K labeled statements from PolitiFact
2. **ISOT** — 44K articles (requires manual Kaggle download)
3. **FakeNewsNet** — PolitiFact + GossipCop articles

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

RAW_DIR = Path('../data/raw')

## 1. LIAR Dataset

12.8K labeled statements from PolitiFact with 6 truthfulness levels.

In [ ]:
# LIAR column names
liar_cols = ['id', 'label', 'statement', 'subject', 'speaker', 'job',
             'state', 'party', 'barely_true_count', 'false_count',
             'half_true_count', 'mostly_true_count', 'pants_fire_count', 'context']

liar_train = pd.read_csv(RAW_DIR / 'liar/train.tsv', sep='\t', header=None, names=liar_cols)
liar_valid = pd.read_csv(RAW_DIR / 'liar/valid.tsv', sep='\t', header=None, names=liar_cols)
liar_test = pd.read_csv(RAW_DIR / 'liar/test.tsv', sep='\t', header=None, names=liar_cols)

print(f'LIAR Train: {liar_train.shape}')
print(f'LIAR Valid: {liar_valid.shape}')
print(f'LIAR Test:  {liar_test.shape}')
print(f'Total:      {len(liar_train) + len(liar_valid) + len(liar_test)}')

liar_all = pd.concat([liar_train, liar_valid, liar_test], ignore_index=True)
liar_all.head()

In [ ]:
# Class distribution
print('=== Label Distribution ===')
print(liar_all['label'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
order = ['pants-fire', 'false', 'barely-true', 'half-true', 'mostly-true', 'true']
colors = ['#d32f2f', '#f44336', '#ff9800', '#ffc107', '#8bc34a', '#4caf50']
liar_all['label'].value_counts().reindex(order).plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('LIAR — Label Distribution')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Pie chart
liar_all['label'].value_counts().reindex(order).plot(kind='pie', ax=axes[1], colors=colors,
                                                      autopct='%1.1f%%')
axes[1].set_title('LIAR — Label Proportions')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Map to binary labels for our use case
label_map = {
    'true': 'real',
    'mostly-true': 'real',
    'half-true': 'real',
    'barely-true': 'fake',
    'false': 'fake',
    'pants-fire': 'fake'
}
liar_all['binary_label'] = liar_all['label'].map(label_map)

print('=== Binary Label Distribution ===')
print(liar_all['binary_label'].value_counts())
print(f'\nRatio: {liar_all["binary_label"].value_counts(normalize=True).to_dict()}')

### Label Mapping Strategy

LIAR uses 6 truthfulness levels. We collapse them into binary labels for our classifier:
- **Real:** `true`, `mostly-true`, `half-true` — statements with substantial truth
- **Fake:** `barely-true`, `false`, `pants-fire` — predominantly misleading statements

The `half-true` threshold is a judgment call — we include it as "real" since these statements contain partial truth and our pipeline's sentiment/credibility signals can catch the nuance.

In [ ]:
# Text length analysis
liar_all['text_length'] = liar_all['statement'].astype(str).str.len()
liar_all['word_count'] = liar_all['statement'].astype(str).str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color in [('real', '#4caf50'), ('fake', '#f44336')]:
    subset = liar_all[liar_all['binary_label'] == label]
    axes[0].hist(subset['text_length'], bins=50, alpha=0.6, label=label, color=color)
    axes[1].hist(subset['word_count'], bins=50, alpha=0.6, label=label, color=color)

axes[0].set_title('LIAR — Statement Character Length')
axes[0].set_xlabel('Characters')
axes[0].legend()
axes[1].set_title('LIAR — Statement Word Count')
axes[1].set_xlabel('Words')
axes[1].legend()

plt.tight_layout()
plt.show()

print(liar_all.groupby('binary_label')[['text_length', 'word_count']].describe())

In [ ]:
# Top speakers and parties
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

liar_all['speaker'].value_counts().head(15).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 Speakers')
axes[0].invert_yaxis()

liar_all['party'].value_counts().head(10).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Top 10 Parties')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Missing values
print('=== Missing Values ===')
print(liar_all.isnull().sum())
print(f'\nDuplicate statements: {liar_all["statement"].duplicated().sum()}')

## 2. ISOT Fake News Dataset

44K articles — Reuters (real) + unreliable sources (fake).

**Note:** Requires manual download from Kaggle. See `data/raw/isot/DOWNLOAD_INSTRUCTIONS.md`.

In [ ]:
isot_true_path = RAW_DIR / 'isot/True.csv'
isot_fake_path = RAW_DIR / 'isot/Fake.csv'

if isot_true_path.exists() and isot_fake_path.exists():
    isot_true = pd.read_csv(isot_true_path)
    isot_true['label'] = 'real'
    isot_fake = pd.read_csv(isot_fake_path)
    isot_fake['label'] = 'fake'
    isot_all = pd.concat([isot_true, isot_fake], ignore_index=True)
    
    print(f'ISOT True: {isot_true.shape}')
    print(f'ISOT Fake: {isot_fake.shape}')
    print(f'Total:     {len(isot_all)}')
    print(f'\nColumns: {list(isot_all.columns)}')
    display(isot_all.head())
    
    # Label distribution
    print(f'\n=== Label Distribution ===')
    print(isot_all['label'].value_counts())
    
    # Text length
    isot_all['text_length'] = isot_all['text'].astype(str).str.len()
    isot_all['word_count'] = isot_all['text'].astype(str).str.split().str.len()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for label, color in [('real', '#4caf50'), ('fake', '#f44336')]:
        subset = isot_all[isot_all['label'] == label]
        axes[0].hist(subset['text_length'], bins=50, alpha=0.6, label=label, color=color)
        axes[1].hist(subset['word_count'], bins=50, alpha=0.6, label=label, color=color)
    axes[0].set_title('ISOT — Article Character Length')
    axes[0].legend()
    axes[1].set_title('ISOT — Article Word Count')
    axes[1].legend()
    plt.tight_layout()
    plt.show()
    
    # Subject distribution
    fig, ax = plt.subplots(figsize=(12, 5))
    isot_all.groupby(['subject', 'label']).size().unstack().plot(kind='bar', ax=ax)
    ax.set_title('ISOT — Subject Distribution by Label')
    plt.tight_layout()
    plt.show()
else:
    print('ISOT dataset not found. Please download from Kaggle:')
    print('https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset')
    print(f'Place True.csv and Fake.csv in: {RAW_DIR / "isot"}')

## 3. FakeNewsNet Dataset

PolitiFact and GossipCop articles with titles and URLs.

In [ ]:
fnn_dir = RAW_DIR / 'fakenewsnet'

fnn_politifact_real = pd.read_csv(fnn_dir / 'politifact_real.csv')
fnn_politifact_fake = pd.read_csv(fnn_dir / 'politifact_fake.csv')
fnn_gossipcop_real = pd.read_csv(fnn_dir / 'gossipcop_real.csv')
fnn_gossipcop_fake = pd.read_csv(fnn_dir / 'gossipcop_fake.csv')

print('=== FakeNewsNet Sizes ===')
print(f'PolitiFact Real: {len(fnn_politifact_real)}')
print(f'PolitiFact Fake: {len(fnn_politifact_fake)}')
print(f'GossipCop Real:  {len(fnn_gossipcop_real)}')
print(f'GossipCop Fake:  {len(fnn_gossipcop_fake)}')

print(f'\nColumns: {list(fnn_politifact_real.columns)}')
fnn_politifact_real.head()

In [ ]:
# Combine FakeNewsNet with labels
fnn_politifact_real['label'] = 'real'
fnn_politifact_fake['label'] = 'fake'
fnn_gossipcop_real['label'] = 'real'
fnn_gossipcop_fake['label'] = 'fake'

fnn_politifact_real['source'] = 'politifact'
fnn_politifact_fake['source'] = 'politifact'
fnn_gossipcop_real['source'] = 'gossipcop'
fnn_gossipcop_fake['source'] = 'gossipcop'

fnn_all = pd.concat([fnn_politifact_real, fnn_politifact_fake,
                      fnn_gossipcop_real, fnn_gossipcop_fake], ignore_index=True)

print(f'Total FakeNewsNet: {len(fnn_all)}')
print(f'\n=== Label Distribution ===')
print(fnn_all['label'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By source
fnn_all.groupby(['source', 'label']).size().unstack().plot(kind='bar', ax=axes[0],
                                                            color=['#f44336', '#4caf50'])
axes[0].set_title('FakeNewsNet — Distribution by Source & Label')
axes[0].tick_params(axis='x', rotation=0)

# Title lengths
fnn_all['title_length'] = fnn_all['title'].astype(str).str.len()
for label, color in [('real', '#4caf50'), ('fake', '#f44336')]:
    subset = fnn_all[fnn_all['label'] == label]
    axes[1].hist(subset['title_length'], bins=50, alpha=0.6, label=label, color=color)
axes[1].set_title('FakeNewsNet — Title Length Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Missing values
print('=== Missing Values ===')
print(fnn_all.isnull().sum())
print(f'\nDuplicate titles: {fnn_all["title"].duplicated().sum()}')
print(f'Null titles: {fnn_all["title"].isna().sum()}')

## 4. Dataset Summary

| Dataset | Real | Fake | Total | Text Column | Notes |
|---------|------|------|-------|-------------|-------|
| LIAR | ~6.4K | ~6.4K | 12,791 | statement | Short statements, 6-class originally |
| ISOT | ~21K | ~23K | ~44K | text | Full articles, needs Kaggle download |
| FakeNewsNet | 17,441 | 5,755 | 23,196 | title | Titles only (no full text) |

## Key Findings

1. **Class balance:** LIAR is roughly balanced (50/50). ISOT has slightly more fake articles. FakeNewsNet is heavily skewed toward real (3:1 ratio in GossipCop).
2. **Text length:** Fake news in LIAR tends to be slightly shorter. ISOT fake articles are comparable in length to real ones. FakeNewsNet only provides titles.
3. **Data quality:** LIAR has minimal missing values. FakeNewsNet has some null titles. ISOT requires manual download from Kaggle.
4. **Combined dataset:** ~80K samples when ISOT is included, providing sufficient data for deep learning (RoBERTa fine-tuning).
5. **Next step:** Preprocess all datasets into a unified format with binary labels (`label_encoded`: 0=real, 1=fake), then split into train/val/test for model training.

In [ ]:
# Summary statistics
print('=== Combined Dataset Summary ===')
print(f'LIAR:         {len(liar_all):>6} samples (statements)')
if isot_true_path.exists():
    print(f'ISOT:         {len(isot_all):>6} samples (full articles)')
else:
    print(f'ISOT:         Not downloaded yet')
print(f'FakeNewsNet:  {len(fnn_all):>6} samples (titles only)')
print(f'\nFor model training, we will primarily use:')
print(f'  - LIAR statements (short text classification)')
print(f'  - ISOT articles (long text classification) — when available')
print(f'  - FakeNewsNet titles (headline classification)')